###  SISTEMA DE QUALIDADE DE DADOS — 6 PILARES
Gera relatório PDF formal a partir de arquivos CSV

Pilares avaliados:
1. Completude    — valores ausentes / nulos
2. Consistência  — duplicatas, conflitos de tipo
3. Conformidade  — formatos esperados (datas, padrões)
4. Integridade   — relacionamentos e valores inválidos
5. Precisão      — outliers e valores suspeitos
6. Atualidade    — quão recentes são os registros

In [ ]:
import sys
import os
import re
import warnings
from datetime import datetime, date
 
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from io import BytesIO
 
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    PageBreak, HRFlowable, Image
)
from reportlab.graphics.shapes import Drawing, Rect, String
from reportlab.graphics import renderPDF
 
warnings.filterwarnings("ignore")

In [ ]:
#  Constantes e configurações globais
CONFIG = {
    # Colunas que NUNCA devem ser nulas (lista de nomes exatos)
    "colunas_obrigatorias": [],
 
    # Colunas que devem ser únicas (ex: CPF, ID)
    "colunas_unicas": [],
 
    # Formatos de data aceitos (o script testa todos automaticamente)
    "formatos_data": ["%d/%m/%Y", "%Y-%m-%d", "%d-%m-%Y", "%Y/%m/%d", "%d.%m.%Y"],
 
    # Colunas que são datas (deixe vazio para detecção automática)
    "colunas_data": [],
 
    # Número de dias: registros mais antigos que isso geram alerta de atualidade
    "atualidade_limite_dias": 365,
 
    # Limiar de outlier via IQR (padrão 1.5)
    "outlier_iqr_fator": 1.5,
 
    # Porcentagem mínima aceitável de completude por coluna (0–100)
    "completude_minima_pct": 80.0,
}
 


#  Paleta de cores
COR_PRIMARIA   = colors.HexColor("#1B3A6B")   # azul escuro
COR_SECUNDARIA = colors.HexColor("#2E86C1")   # azul médio
COR_DESTAQUE   = colors.HexColor("#F39C12")   # âmbar
COR_SUCESSO    = colors.HexColor("#27AE60")   # verde
COR_ALERTA     = colors.HexColor("#E74C3C")   # vermelho
COR_NEUTRO     = colors.HexColor("#ECF0F1")   # cinza claro
COR_TEXTO      = colors.HexColor("#2C3E50")   # quase preto

NameError: name 'colors' is not defined

###  1. LEITURA E PRÉ-PROCESSAMENTO

In [ ]:
def carregar_csv(caminho: str) -> pd.DataFrame:
    """Tenta ler o CSV com diferentes encodings e separadores."""
    encodings = ["utf-8", "latin-1", "cp1252", "utf-8-sig"]
    separadores = [",", ";", "\t", "|"]
    for enc in encodings:
        for sep in separadores:
            try:
                df = pd.read_csv(caminho, encoding=enc, sep=sep, low_memory=False)
                if df.shape[1] > 1:
                    return df
            except Exception:
                continue
    raise ValueError(f"Não foi possível ler o arquivo: {caminho}")
 
 
def detectar_colunas_data(df: pd.DataFrame) -> list:
    """Detecta automaticamente colunas que parecem conter datas."""
    candidatas = []
    palavras_chave = ["data", "date", "dt_", "_dt", "vencimento", "nascimento",
                      "criacao", "criação", "abertura", "fechamento", "prazo"]
    for col in df.columns:
        col_lower = col.lower()
        if any(k in col_lower for k in palavras_chave):
            candidatas.append(col)
            continue
        # Testa amostra da coluna
        amostra = df[col].dropna().astype(str).head(20)
        for fmt in CONFIG["formatos_data"]:
            try:
                pd.to_datetime(amostra, format=fmt, errors="raise")
                candidatas.append(col)
                break
            except Exception:
                continue
    return list(set(candidatas))
 
 
def tentar_parse_data(serie: pd.Series) -> pd.Series:
    """Tenta converter uma série para datetime usando os formatos configurados."""
    for fmt in CONFIG["formatos_data"]:
        try:
            return pd.to_datetime(serie, format=fmt, errors="coerce")
        except Exception:
            continue
    return pd.to_datetime(serie, infer_datetime_format=True, errors="coerce")